# 向量

In [4]:
import numpy as np

In [5]:
class Vector:
           def __init__(self, components):
               self.components = list(components)
               self.dim = len(self.components)

           def __add__(self, other):
               # 向量加法：对应维度逐元素相加
               return Vector([a + b for a, b in zip(self.components, other.components)])
           def __sub__(self, other):
               # 向量减法: 对应维度逐元素相减
               return Vector([a - b for a, b in zip(self.components, other.components)])

           def dot(self, other):
               # 点积：对应分量相乘，然后全局累加求和
               return sum(a * b for a, b in zip(self.components, other.components))

           def magnitude(self):
               # 模长（L2 范数）：各分量平方和的开根号
               return sum(x**2 for x in self.components) ** 0.5

           def normalize(self):
               # 归一化：将向量缩放为模长为 1 的单位向量
               mag = self.magnitude()
               return Vector([x / mag for x in self.components])

           def cosine_similarity(self, other):
               # 余弦相似度：点积 / (模长乘积)
               return self.dot(other) / (self.magnitude() * other.magnitude())
           def project_onto(self, other):
                # 投影公式：proj_b(a) = (a · b / b · b) * b
                scalar = self.dot(other) / other.dot(other)
                return Vector([scalar * x for x in other.components])
           def angle_between(self,other):
                angle_rad = np.arccos(self.cosine_similarity(other))  # 得到弧度
                angle_deg = np.degrees(angle_rad) # 转换为角度

                return angle_deg
            

In [6]:
v1 = Vector([3, 0])
v2 = Vector([2, 0])

v1.magnitude()
print(v2.normalize().components)
v1.dot(v2)

[1.0, 0.0]


6

In [7]:
def project_onto(self, other):
    # 投影公式：proj_b(a) = (a · b / b · b) * b
    scalar = self.dot(other) / other.dot(other)
    return Vector([scalar * x for x in other.components])

In [8]:
a = Vector([3, 4])
b = Vector([1, 0])
proj = a.project_onto(b)
print(proj.components)
residual = a - proj
print(residual.components)
print(residual.dot(b))


[3.0, 0.0]
[0.0, 4.0]
0.0


# Gram-Schmidt 正交化

In [9]:
def gram_schmidt(vectors):
    orthonormal = []
    for v in vectors:
        w = v
        # 核心：从当前向量中，逐一把在已有基向量上的“投影影子”全部减掉！
        for u in orthonormal:
            proj = w.project_onto(u)
            w = w - proj
        # 如果扣除完已经退化为零向量（说明线性相关），跳过
        if w.magnitude() < 1e-10:
            continue
        # 剩下的是完全正交的新残差，归一化成长度为 1 的基向量
        orthonormal.append(w.normalize())
    return orthonormal

In [10]:
v3 = Vector([1, 1])
v4 = Vector([1, 0])
u3 = v3.normalize()
print(u3.components)
print(v4.project_onto(u3).components)
residual = v4 - v4.project_onto(u3)
print(residual.components)


[0.7071067811865475, 0.7071067811865475]
[0.5, 0.5]
[0.5, -0.5]


# 矩阵

In [37]:
class Matrix:
    def __init__(self, rows):
        self.rows = [list(row) for row in rows]
        self.shape = (len(self.rows), len(self.rows[0]))  # (行数 m, 列数 n)

    def __matmul__(self, other):
        # 模式一：矩阵乘向量 (m x n) @ (n 维向量) -> (m 维新向量)
        if isinstance(other, Vector):
            return Vector([
                sum(self.rows[i][j] * other.components[j] for j in range(self.shape[1]))
                for i in range(self.shape[0])
            ])

        # 模式二：矩阵乘矩阵 (m x k) @ (k x p) -> (m x p)
        rows = []
        for i in range(self.shape[0]):
            row = []
            for j in range(other.shape[1]):
                row.append(sum(
                    self.rows[i][k] * other.rows[k][j]
                    for k in range(self.shape[1])
                ))
            rows.append(row)
        return Matrix(rows)

    def rank(self):
        rows = [row[:] for row in self.rows]
        m, n = self.shape
        r = 0
        for col in range(n):
            # 1. 寻找主元（Pivot）
            pivot = None
            for row in range(r, m):
                if abs(rows[row][col]) > 1e-10:
                    pivot = row
                    break
            if pivot is None:
                continue
            # 2. 交换行并把主元归一化为 1
            rows[r], rows[pivot] = rows[pivot], rows[r]
            scale = rows[r][col]
            rows[r] = [x / scale for x in rows[r]]
            # 3. 将其他所有行的该列元素消为 0
            for row in range(m):
                if row != r and abs(rows[row][col]) > 1e-10:
                    factor = rows[row][col]
                    rows[row] = [rows[row][j] - factor * rows[r][j] for j in range(n)]
            r += 1
        return r
        


In [12]:
R = Matrix([[0,-1],[1,0]])

point = Vector([3,1])
rotated = R @ point
print(rotated.components)

[-1, 3]


# Rank

In [36]:
def rank(self):
    rows = [row[:] for row in self.rows]
    m, n = self.shape
    r = 0
    for col in range(n):
        # 1. 寻找主元（Pivot）
        pivot = None
        for row in range(r, m):
            if abs(rows[row][col]) > 1e-10:
                pivot = row
                break
        if pivot is None:
            continue
        # 2. 交换行并把主元归一化为 1
        rows[r], rows[pivot] = rows[pivot], rows[r]
        scale = rows[r][col]
        rows[r] = [x / scale for x in rows[r]]
        # 3. 将其他所有行的该列元素消为 0
        for row in range(m):
            if row != r and abs(rows[row][col]) > 1e-10:
                factor = rows[row][col]
                rows[row] = [rows[row][j] - factor * rows[r][j] for j in range(n)]
        r += 1
    return r


In [14]:
M = Matrix([[1,2],[2,4]])
M.rank()

1

In [15]:
import numpy as np


In [16]:
a = np.array([1, 2, 3], dtype=float)
b = np.array([4, 5, 6], dtype=float)
dot_val = np.dot(a,b)
norm_a = np.linalg.norm(a)
cos_sim = dot_val / (norm_a * np.linalg.norm(b))

A = np.array([[1, 2], [2, 4]])
print(f"Rank: {np.linalg.matrix_rank(A)}")

Q, R = np.linalg.qr(np.random.randn(3, 3))
print("Q 正交检验 (Q @ Q.T == I):", np.allclose(Q @ Q.T, np.eye(3)))

Rank: 1
Q 正交检验 (Q @ Q.T == I): True


In [17]:
import torch

x = torch.randn(3, requires_grad=True)
y = torch.tensor([1.0,0.0,0.0])

similarity = torch.dot(x,y)

similarity.backward()

print("x 的梯度 d(similarity)/dx:", x.grad)

x 的梯度 d(similarity)/dx: tensor([1., 0., 0.])


# test

In [18]:
# 1.实现 Vector.angle_between(other)，返回两个向量夹角（度）
def angle_between(self,other):
    angle_rad = np.arccos(self.cosine_similarity(other))  # 得到弧度
    angle_deg = np.degrees(angle_rad) # 转换为角度

    return angle_deg
v1 = Vector([1, 0])
v2 = Vector([-1, 0])
v1.angle_between(v2)


np.float64(180.0)

In [19]:
# 2.做一个二维缩放矩阵：x 变 2 倍、y 变 3 倍，作用在向量 [1, 1] 上
M1 = Matrix([[2,0],[0,3]])
v3 = Vector([1, 1])
print((M1 @ v3).components)


[2, 3]


In [20]:
# 3.给定 5 个随机的「词向量」（维度 50），用 cosine similarity 找出最相似的一对
import numpy as np

# 生成 5 个 50 维词向量
word_vectors = np.random.randn(5, 50)

# 计算余弦相似度矩阵
V = word_vectors / np.linalg.norm(word_vectors, axis=1, keepdims=True)
similarity_matrix = V @ V.T

# 方法1
# 排除自己与自己的相似度
scores = similarity_matrix.copy()
np.fill_diagonal(scores, -np.inf)

# 找到相似度最高的一对
i, j = np.unravel_index(np.argmax(scores), scores.shape)

print("最相似的两个词向量：", i, j)
print("余弦相似度：", similarity_matrix[i, j])

# 方法2
i, j = np.triu_indices(5, k=1)
idx = np.argmax(similarity_matrix[i, j])
print(idx)
print(i[idx], j[idx], similarity_matrix[i[idx], j[idx]])

最相似的两个词向量： 0 2
余弦相似度： 0.13261751451251716
1
0 2 0.13261751451251716


In [31]:
# 4.验证 Gram-Schmidt 的输出真是标准正交：每对点积为 0，每个长度为 1
import numpy as np

# 创建 5 个 50 维随机向量
A = np.random.randn(5, 50)

# 正交化
Q, R = np.linalg.qr(A.T)

# 转置回来，每一行是一个正交单位向量
Q = Q.T

print(Q.shape)  # (5, 50)
print(np.allclose(Q @ Q.T, np.eye(5)))  # True


(5, 50)
True


In [39]:
# 5.做一个秩为 2 的 3×3 矩阵。用 rank() 验证。然后说明这些列在几何上张成什么
B = Matrix([[1,2,3],[1,2,4],[2,4,6]])
B.rank()

2

In [ ]:
# 6.把 [1, 2, 3] 投影到 [1, 1, 1] 上。结果在几何上代表什么？
c = Vector([1, 2, 3])
d = Vector([1, 1, 1])
c.project_onto(d).components 
# 代表[1, 2, 3]在这个[1, 1,]

[2.0, 2.0, 2.0]